In [ ]:
# Imbalance handling - without using Pipeline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from imblearn.under_sampling import EditedNearestNeighbours, TomekLinks
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ---------------------------------------------------------
# 1. Read Data
# ---------------------------------------------------------
# Read the file directly into a pandas DataFrame without dummy data or exception handling
df = pd.read_csv('clusters-3-imbalanced.csv')

# Separate features (X) and target variable (y)
target_col = 'yclass'
feature_cols = [col for col in df.columns if col != target_col]

X = df[feature_cols].values
y = df[target_col].values

# ---------------------------------------------------------
# 2. Vertical Frequency Plot of 'yclass'
# ---------------------------------------------------------
class_counts = df[target_col].value_counts().sort_index()

plt.figure(figsize=(6, 5))
plt.bar(class_counts.index.astype(str), class_counts.values, color='steelblue', edgecolor='black')
plt.title("Vertical Frequency Plot of 'yclass'")
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 3. Train/Test Split & SMOTE Balancing
# ---------------------------------------------------------
# Stratified split ensures representation of each class in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Apply ENN, Tomek Links, and SMOTE sequentially on training data
enn = EditedNearestNeighbours()
X_enn, y_enn = enn.fit_resample(X_train, y_train)

tl = TomekLinks()
X_tl, y_tl = tl.fit_resample(X_enn, y_enn)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_tl, y_tl)

# ---------------------------------------------------------
# 4. Define Classifiers
# ---------------------------------------------------------
classifiers = [
    LogisticRegression(max_iter=1000, random_state=42),
    RandomForestClassifier(n_estimators=100, random_state=42),
    CatBoostClassifier(verbose=0, random_state=42)
]

# ---------------------------------------------------------
# Helper Function: Plot Decision Boundary
# ---------------------------------------------------------
def plot_decision_boundary(model, X_data, y_data, title, ax):
    x_min, x_max = X_data[:, 0].min() - 1, X_data[:, 0].max() + 1
    y_min, y_max = X_data[:, 1].min() - 1, X_data[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points)

    if Z.ndim > 1:
        Z = Z.ravel()

    classes, inverse_indices = np.unique(Z, return_inverse=True)
    Z_grid = inverse_indices.reshape(xx.shape)

    ax.contourf(xx, yy, Z_grid, alpha=0.3, cmap=plt.cm.coolwarm)
    
    ax.scatter(
        X_data[:, 0],
        X_data[:, 1],
        c=y_data,
        cmap=plt.cm.coolwarm,
        edgecolor='k',
        s=30
    )
    ax.set_title(title)
    ax.set_xlabel(feature_cols[0])
    ax.set_ylabel(feature_cols[1])

# ---------------------------------------------------------
# 5. Model Training, Evaluation Metrics, and Boundary Plotting
# ---------------------------------------------------------
metrics_records = []
classes = np.unique(y)

for clf in classifiers:
    clf_name = clf.__class__.__name__

    # Train model on SMOTE-resampled training data
    clf.fit(X_train_resampled, y_train_resampled)

    # Predictions
    y_train_pred = clf.predict(X_train_resampled)
    y_test_pred = clf.predict(X_test)

    if hasattr(y_train_pred, 'ravel'):
        y_train_pred = y_train_pred.ravel()
    if hasattr(y_test_pred, 'ravel'):
        y_test_pred = y_test_pred.ravel()

    # Per-class metric calculations
    train_p, train_r, train_f1, _ = precision_recall_fscore_support(
        y_train_resampled, y_train_pred, labels=classes, zero_division=0
    )
    test_p, test_r, test_f1, _ = precision_recall_fscore_support(
        y_test, y_test_pred, labels=classes, zero_division=0
    )

    # Base record with classifier name as the first column
    record = {
        'Classifier': clf_name,
        'Train Accuracy': accuracy_score(y_train_resampled, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred)
    }

    # Add per-class metrics dynamically
    for idx, cls in enumerate(classes):
        record[f'Train Precision (Class {cls})'] = train_p[idx]
        record[f'Test Precision (Class {cls})'] = test_p[idx]
        record[f'Train Recall (Class {cls})'] = train_r[idx]
        record[f'Test Recall (Class {cls})'] = test_r[idx]
        record[f'Train F1 (Class {cls})'] = train_f1[idx]
        record[f'Test F1 (Class {cls})'] = test_f1[idx]

    metrics_records.append(record)

    # Plot decision boundaries for Train (SMOTE) and Test
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_decision_boundary(clf, X_train_resampled, y_train_resampled, f"{clf_name} - Train Decision Boundary (ENN + TL + SMOTE)", axes[0])
    plot_decision_boundary(clf, X_test, y_test, f"{clf_name} - Test Decision Boundary", axes[1])
    plt.tight_layout()
    plt.show()

# Convert metrics list to DataFrame
metrics_df = pd.DataFrame(metrics_records)

# Display summary table of metrics
print("\nClassification Metrics Summary (Per-Class):")
print(metrics_df.T.to_string())

# ---------------------------------------------------------
# 6. Vertical Grouped Bar Plots for Metrics
# ---------------------------------------------------------
classifiers_list = metrics_df['Classifier'].tolist()
split_types = ['Train', 'Test']
metric_names = ['Precision', 'Recall', 'F1']

fig, axes = plt.subplots(len(metric_names), len(split_types), figsize=(14, 12), sharey=True)

x = np.arange(len(classes))
bar_width = 0.8 / len(classifiers_list)

for row_idx, metric in enumerate(metric_names):
    for col_idx, split in enumerate(split_types):
        ax = axes[row_idx, col_idx]
        
        for clf_idx, clf_name in enumerate(classifiers_list):
            clf_row = metrics_df[metrics_df['Classifier'] == clf_name]
            values = [clf_row[f'{split} {metric} (Class {cls})'].values[0] for cls in classes]
            offsets = x + (clf_idx - (len(classifiers_list) - 1) / 2) * bar_width
            ax.bar(offsets, values, width=bar_width, label=clf_name, edgecolor='black', alpha=0.85)

        ax.set_title(f"{split} - {metric} per Class", fontsize=11, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([f"Class {cls}" for cls in classes])
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        if row_idx == 0 and col_idx == 1:
            ax.legend(title="Classifier", bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
# Imbalance handling - Pipeline implementation

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import EditedNearestNeighbours, TomekLinks
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

# ---------------------------------------------------------
# 1. Read Data
# ---------------------------------------------------------
df = pd.read_csv('clusters-3-imbalanced.csv')

target_col = 'yclass'
feature_cols = [col for col in df.columns if col != target_col]

X = df[feature_cols].values
y = df[target_col].values

# ---------------------------------------------------------
# 2. Vertical Frequency Plot of 'yclass'
# ---------------------------------------------------------
class_counts = df[target_col].value_counts().sort_index()

plt.figure(figsize=(6, 5))
plt.bar(class_counts.index.astype(str), class_counts.values, color='steelblue', edgecolor='black')
plt.title("Vertical Frequency Plot of 'yclass'")
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 3. Train/Test Split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# 4. Define Pipelines
# ---------------------------------------------------------
classifiers = [
    LogisticRegression(max_iter=1000, random_state=42),
    RandomForestClassifier(n_estimators=100, random_state=42),
    CatBoostClassifier(verbose=0, random_state=42)
]

# Build imblearn Pipelines combining resampling and model estimation
# Comment / uncomment lines to see the effects of each / combined imbalance handling methods

pipelines = [
    Pipeline([
        ('smote', SMOTE(random_state=42)),
#        ('tomek', TomekLinks()),
        ('enn', EditedNearestNeighbours()),
        ('classifier', clf)
    ])
    for clf in classifiers
]

# ---------------------------------------------------------
# Helper Function: Plot Decision Boundary
# ---------------------------------------------------------
def plot_decision_boundary(model, X_data, y_data, title, ax):
    x_min, x_max = X_data[:, 0].min() - 1, X_data[:, 0].max() + 1
    y_min, y_max = X_data[:, 1].min() - 1, X_data[:, 1].max() + 1
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points)

    if Z.ndim > 1:
        Z = Z.ravel()

    classes, inverse_indices = np.unique(Z, return_inverse=True)
    Z_grid = inverse_indices.reshape(xx.shape)

    ax.contourf(xx, yy, Z_grid, alpha=0.3, cmap=plt.cm.coolwarm)
    
    ax.scatter(
        X_data[:, 0],
        X_data[:, 1],
        c=y_data,
        cmap=plt.cm.coolwarm,
        edgecolor='k',
        s=30
    )
    ax.set_title(title)
    ax.set_xlabel(feature_cols[0])
    ax.set_ylabel(feature_cols[1])

# ---------------------------------------------------------
# 5. Model Training, Evaluation Metrics, and Boundary Plotting
# ---------------------------------------------------------
metrics_records = []
classes = np.unique(y)

for pipe in pipelines:
    clf_name = pipe.named_steps['classifier'].__class__.__name__

    # Train entire pipeline on raw training split
    pipe.fit(X_train, y_train)

    # Predictions (Pipeline bypasses samplers during .predict())
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    if hasattr(y_train_pred, 'ravel'):
        y_train_pred = y_train_pred.ravel()
    if hasattr(y_test_pred, 'ravel'):
        y_test_pred = y_test_pred.ravel()

    # Per-class metric calculations on raw splits
    train_p, train_r, train_f1, _ = precision_recall_fscore_support(
        y_train, y_train_pred, labels=classes, zero_division=0
    )
    test_p, test_r, test_f1, _ = precision_recall_fscore_support(
        y_test, y_test_pred, labels=classes, zero_division=0
    )

    # Base record
    record = {
        'Classifier': clf_name,
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Test Accuracy': accuracy_score(y_test, y_test_pred)
    }

    # Add per-class metrics dynamically
    for idx, cls in enumerate(classes):
        record[f'Train Precision (Class {cls})'] = train_p[idx]
        record[f'Test Precision (Class {cls})'] = test_p[idx]
        record[f'Train Recall (Class {cls})'] = train_r[idx]
        record[f'Test Recall (Class {cls})'] = test_r[idx]
        record[f'Train F1 (Class {cls})'] = train_f1[idx]
        record[f'Test F1 (Class {cls})'] = test_f1[idx]

    metrics_records.append(record)

    # Plot decision boundaries using the fitted pipeline directly
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    plot_decision_boundary(pipe, X_train, y_train, f"{clf_name} - Train Decision Boundary", axes[0])
    plot_decision_boundary(pipe, X_test, y_test, f"{clf_name} - Test Decision Boundary", axes[1])
    plt.tight_layout()
    plt.show()

# Convert metrics list to DataFrame
metrics_df = pd.DataFrame(metrics_records)

# Display summary table of metrics
print("\nClassification Metrics Summary (Per-Class):")
print(metrics_df.T.to_string())

# ---------------------------------------------------------
# 6. Vertical Grouped Bar Plots for Metrics
# ---------------------------------------------------------
classifiers_list = metrics_df['Classifier'].tolist()
split_types = ['Train', 'Test']
metric_names = ['Precision', 'Recall', 'F1']

fig, axes = plt.subplots(len(metric_names), len(split_types), figsize=(14, 12), sharey=True)

x = np.arange(len(classes))
bar_width = 0.8 / len(classifiers_list)

for row_idx, metric in enumerate(metric_names):
    for col_idx, split in enumerate(split_types):
        ax = axes[row_idx, col_idx]
        
        for clf_idx, clf_name in enumerate(classifiers_list):
            clf_row = metrics_df[metrics_df['Classifier'] == clf_name]
            values = [clf_row[f'{split} {metric} (Class {cls})'].values[0] for cls in classes]
            offsets = x + (clf_idx - (len(classifiers_list) - 1) / 2) * bar_width
            ax.bar(offsets, values, width=bar_width, label=clf_name, edgecolor='black', alpha=0.85)

        ax.set_title(f"{split} - {metric} per Class", fontsize=11, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels([f"Class {cls}" for cls in classes])
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1.05)
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        if row_idx == 0 and col_idx == 1:
            ax.legend(title="Classifier", bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()